# R Master v4 · Mona_Main 结构扫描（一键版）

这一步**不改模型、不删任何面、不渲染大图**，只做结构体检。

它直接读取：
`MyDrive/R_Master/v2/latest/R_Master_Align_v2_PREVIEW.blend`

会输出：
- `R_Master_v4_structure.json`：完整结构报告
- `R_Master_v4_materials.csv`：每个材质槽对应的面数、顶点数、包围盒
- `R_Master_v4_components.csv`：每个连通几何块的顶点/面数、材质组成、位置和尺寸
- `R_Master_v4_Report.zip`：手机只下载这个小包

目的：先确认 Mona_Main 里哪些是真身体、哪些是衣物/鞋/配件，再决定 v5 怎么安全剥离。


In [ ]:
from google.colab import drive, files
from pathlib import Path
import shutil, subprocess, json, zipfile, csv, os

print("R Master v4 · Mona_Main 结构扫描")
drive.mount("/content/drive")

ROOT=Path("/content/drive/MyDrive/R_Master")
SRC=ROOT/"v2"/"latest"/"R_Master_Align_v2_PREVIEW.blend"
CACHE=ROOT/"cache"
OUT=ROOT/"v4_structure"/"latest"
CACHE.mkdir(parents=True,exist_ok=True)
OUT.mkdir(parents=True,exist_ok=True)

if not SRC.exists() or SRC.stat().st_size < 50*1024*1024:
    raise RuntimeError("没找到 v2 预览文件：MyDrive/R_Master/v2/latest/R_Master_Align_v2_PREVIEW.blend")

print(f"✓ v2 源：{SRC.stat().st_size/1024/1024:.1f} MiB")

BLENDER_VERSION="4.4.3"
BLENDER_URL="https://download.blender.org/release/Blender4.4/blender-4.4.3-linux-x64.tar.xz"
LOCAL=Path("/content/r_master_v4")
LOCAL.mkdir(parents=True,exist_ok=True)
ARCHIVE=LOCAL/f"blender-{BLENDER_VERSION}-linux-x64.tar.xz"
BDIR=LOCAL/f"blender-{BLENDER_VERSION}-linux-x64"
DRIVE_ARCHIVE=CACHE/ARCHIVE.name

if shutil.which("xvfb-run") is None:
    subprocess.run(["apt-get","update","-qq"],check=True,stdout=subprocess.DEVNULL,stderr=subprocess.STDOUT)
    subprocess.run(["apt-get","install","-y","-qq","xvfb","libgl1","libx11-6","libxi6","libxrender1","libxfixes3","libxkbcommon0","libsm6"],check=True,stdout=subprocess.DEVNULL,stderr=subprocess.STDOUT)

if DRIVE_ARCHIVE.exists() and DRIVE_ARCHIVE.stat().st_size > 100*1024*1024:
    shutil.copy2(DRIVE_ARCHIVE,ARCHIVE)
    print("✓ 复用 Blender Drive 缓存")
else:
    print("补下载 Blender 一次…")
    subprocess.run(["wget","-q","--show-progress","-O",str(ARCHIVE),BLENDER_URL],check=True)
    shutil.copy2(ARCHIVE,DRIVE_ARCHIVE)

if not (BDIR/"blender").exists():
    if BDIR.exists(): shutil.rmtree(BDIR)
    subprocess.run(["tar","-xf",str(ARCHIVE),"-C",str(LOCAL)],check=True)
BLENDER=BDIR/"blender"
print("✓ Blender 就绪")

SCRIPT=LOCAL/"R_Master_v4_Scan.py"
SCRIPT.write_text(r"""
import bpy, os, sys, json, csv
from collections import defaultdict, Counter, deque
from mathutils import Vector

argv=sys.argv[sys.argv.index("--")+1:] if "--" in sys.argv else []
out=None
for i,a in enumerate(argv):
    if a=="--out" and i+1<len(argv):
        out=argv[i+1]
if not out:
    raise RuntimeError("missing --out")
os.makedirs(out,exist_ok=True)

body=bpy.data.objects.get("R2_Mona_Main") or bpy.data.objects.get("Mona_Main")
if not body or body.type!="MESH":
    raise RuntimeError("找不到 R2_Mona_Main / Mona_Main")

mesh=body.data
verts=mesh.vertices
polys=mesh.polygons
edges=mesh.edges
mw=body.matrix_world

def bbox_from_vertex_ids(ids):
    pts=[mw @ verts[i].co for i in ids]
    if not pts:
        return None
    mn=Vector((min(p.x for p in pts),min(p.y for p in pts),min(p.z for p in pts)))
    mx=Vector((max(p.x for p in pts),max(p.y for p in pts),max(p.z for p in pts)))
    c=(mn+mx)*0.5
    e=mx-mn
    return {
        "min":[float(x) for x in mn],
        "max":[float(x) for x in mx],
        "center":[float(x) for x in c],
        "extent":[float(x) for x in e],
    }

# material slot scan
mat_face_ids=defaultdict(list)
mat_vert_ids=defaultdict(set)
for p in polys:
    mat_face_ids[p.material_index].append(p.index)
    for vi in p.vertices:
        mat_vert_ids[p.material_index].add(int(vi))

materials=[]
for idx in range(max(len(body.material_slots), (max(mat_face_ids.keys())+1 if mat_face_ids else 0))):
    slot=body.material_slots[idx] if idx < len(body.material_slots) else None
    mat=slot.material if slot else None
    name=(mat.name if mat else (slot.name if slot else f"slot_{idx}"))
    vids=mat_vert_ids.get(idx,set())
    fids=mat_face_ids.get(idx,[])
    materials.append({
        "slot":idx,
        "name":name,
        "face_count":len(fids),
        "vertex_count":len(vids),
        "bbox":bbox_from_vertex_ids(vids),
    })

# vertex adjacency from edges
adj=[[] for _ in range(len(verts))]
for e in edges:
    a,b=int(e.vertices[0]),int(e.vertices[1])
    adj[a].append(b); adj[b].append(a)

used=[False]*len(verts)
components=[]
comp_id=0

# helper: polygons by vertex for fast assignment
vert_polys=[[] for _ in range(len(verts))]
for p in polys:
    for vi in p.vertices:
        vert_polys[int(vi)].append(p.index)

for seed in range(len(verts)):
    if used[seed]:
        continue
    used[seed]=True
    q=deque([seed])
    comp_verts=[]
    while q:
        v=q.popleft()
        comp_verts.append(v)
        for n in adj[v]:
            if not used[n]:
                used[n]=True
                q.append(n)

    comp_vset=set(comp_verts)
    comp_faces=set()
    for v in comp_verts:
        comp_faces.update(vert_polys[v])

    # keep only polygons fully inside component
    comp_faces=[pi for pi in comp_faces if all(int(vi) in comp_vset for vi in polys[pi].vertices)]
    hist=Counter(int(polys[pi].material_index) for pi in comp_faces)
    bbox=bbox_from_vertex_ids(comp_verts)
    material_hist=[
        {
            "slot":mi,
            "name":materials[mi]["name"] if mi < len(materials) else f"slot_{mi}",
            "face_count":cnt,
        }
        for mi,cnt in hist.most_common()
    ]
    components.append({
        "component_id":comp_id,
        "vertex_count":len(comp_verts),
        "face_count":len(comp_faces),
        "material_histogram":material_hist,
        "bbox":bbox,
    })
    comp_id+=1

components.sort(key=lambda x:(x["face_count"],x["vertex_count"]), reverse=True)

# assign stable rank after sorting
for rank,c in enumerate(components,1):
    c["rank_by_face_count"]=rank

# whole body bbox
all_bbox=bbox_from_vertex_ids(range(len(verts)))

report={
    "ok":True,
    "stage":"R_Master_v4_MonaMainStructureScan",
    "blend_file":bpy.data.filepath,
    "body_object":body.name,
    "mesh_data":mesh.name,
    "vertex_count":len(verts),
    "edge_count":len(edges),
    "face_count":len(polys),
    "material_slot_count":len(body.material_slots),
    "component_count":len(components),
    "whole_bbox":all_bbox,
    "materials":materials,
    "components":components,
    "notes":[
        "No geometry was deleted or modified.",
        "Material slots and connected components are diagnostic evidence only.",
        "v5 should choose hide/delete targets only after reviewing this report."
    ]
}

json_path=os.path.join(out,"R_Master_v4_structure.json")
with open(json_path,"w",encoding="utf-8") as f:
    json.dump(report,f,ensure_ascii=False,indent=2)

mat_csv=os.path.join(out,"R_Master_v4_materials.csv")
with open(mat_csv,"w",newline="",encoding="utf-8-sig") as f:
    w=csv.writer(f)
    w.writerow(["slot","name","face_count","vertex_count","center_x","center_y","center_z","extent_x","extent_y","extent_z"])
    for m in materials:
        b=m["bbox"] or {"center":[None]*3,"extent":[None]*3}
        w.writerow([m["slot"],m["name"],m["face_count"],m["vertex_count"],*b["center"],*b["extent"]])

comp_csv=os.path.join(out,"R_Master_v4_components.csv")
with open(comp_csv,"w",newline="",encoding="utf-8-sig") as f:
    w=csv.writer(f)
    w.writerow(["rank","component_id","face_count","vertex_count","center_x","center_y","center_z","extent_x","extent_y","extent_z","top_materials"])
    for c in components:
        b=c["bbox"] or {"center":[None]*3,"extent":[None]*3}
        mats=" | ".join([f'{x["slot"]}:{x["name"]}({x["face_count"]})' for x in c["material_histogram"][:8]])
        w.writerow([c["rank_by_face_count"],c["component_id"],c["face_count"],c["vertex_count"],*b["center"],*b["extent"],mats])

print("[R Master v4] SCAN_OK")
print("[R Master v4] body:",body.name)
print("[R Master v4] verts/edges/faces:",len(verts),len(edges),len(polys))
print("[R Master v4] materials:",len(materials))
print("[R Master v4] components:",len(components))
print("[R Master v4] top components:",[(c["rank_by_face_count"],c["face_count"],c["vertex_count"],c["material_histogram"][:3]) for c in components[:12]])
""",encoding="utf-8")

for p in OUT.iterdir():
    if p.is_file(): p.unlink()

print("③ 扫描 Mona_Main 内部结构…")
LOG=OUT/"R_Master_v4_blender.log"
cmd=["xvfb-run","-a",str(BLENDER),"--background",str(SRC),"--python",str(SCRIPT),"--","--out",str(OUT)]
with LOG.open("w",encoding="utf-8") as log:
    p=subprocess.Popen(cmd,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
    for line in p.stdout:
        log.write(line)
        if "R Master v4" in line or "Traceback" in line or "Error" in line:
            print(line.rstrip())
    rc=p.wait()
if rc!=0:
    print(LOG.read_text(encoding="utf-8",errors="replace")[-9000:])
    raise RuntimeError(f"v4 扫描失败，退出码 {rc}。截图给二蛋即可。")

required=[
    OUT/"R_Master_v4_structure.json",
    OUT/"R_Master_v4_materials.csv",
    OUT/"R_Master_v4_components.csv",
    LOG
]
missing=[p.name for p in required if not p.exists()]
if missing:
    raise RuntimeError("v4 缺少输出："+", ".join(missing))

report=json.loads((OUT/"R_Master_v4_structure.json").read_text(encoding="utf-8"))
print("\n✓ R Master v4 SCAN_OK")
print("  Body:",report["body_object"])
print("  顶点 / 面：",report["vertex_count"],"/",report["face_count"])
print("  材质槽：",report["material_slot_count"])
print("  连通组件：",report["component_count"])

print("\n最大的 12 个连通组件：")
for c in report["components"][:12]:
    mats=", ".join([f'{x["name"]}:{x["face_count"]}' for x in c["material_histogram"][:4]])
    print(f'  #{c["rank_by_face_count"]} faces={c["face_count"]} verts={c["vertex_count"]} · {mats}')

ZIP=OUT/"R_Master_v4_Report.zip"
with zipfile.ZipFile(ZIP,"w",compression=zipfile.ZIP_DEFLATED,compresslevel=6) as z:
    for p in required:
        z.write(p,arcname=p.name)

print(f"\n✓ 报告包：{ZIP.stat().st_size/1024:.1f} KiB")
files.download(str(ZIP))

